# 安装

#安装参考：https://github.com/OpenBMB/MiniCPM-V/blob/main/README_zh.md#%E5%AE%89%E8%A3%85

#第一步：下载模型https://modelscope.cn/models/OpenBMB/MiniCPM-Llama3-V-2_5-int4/files

    在cmd中运行该代码：

    git clone https://www.modelscope.cn/OpenBMB/MiniCPM-Llama3-V-2_5-int4.git

#第二步：

    cd MiniCPM-Llama3-V-2_5-int4
    
    MiniCPM-Llama3-V-2_5-int4>conda create -n MiniCPMV python=3.10 -y

#第三步：激活虚拟环境：

    conda activate MiniCPMV

#第四步：安装依赖包：

    pip install Pillow==10.1.0 transformers==4.40.0 sentencepiece==0.1.99 accelerate==0.30.1 bitsandbytes==0.43.1
    
    #单独安装CUDA版本的Pytorch: pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    
    #Successfully installed torch-2.3.1+cu121 torchaudio-2.3.1+cu121 torchvision-0.18.1+cu121
    
#第五步：安装ipykernel：

    conda activate MiniCPMV  

    pip install ipykernel

    python -m ipykernel install --user --name MiniCPMV --display-name "MiniCPMV"


import os
os.chdir('C:/Users/yurul/MiniCPM-Llama3-V-2_5-int4')

# 测试

In [ ]:
import torch
from PIL import Image
import warnings
from transformers import AutoModel, AutoTokenizer

In [ ]:
# 指定本地模型目录
model_dir = 'C:/Users/yurul/MiniCPM-Llama3-V-2_5-int4'

In [ ]:
model = AutoModel.from_pretrained(model_dir, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
#model.eval()

In [ ]:
# 忽略Flash Attention相关的警告
warnings.filterwarnings("ignore", message=".*not compiled with flash attention.*")

In [ ]:
image = Image.open(r'F:\phd data\post images\cem.oezdemir_CRhCt9eqBu4.jpg').convert('RGB')
question = 'Whether to include the German Green Party logo, which is a hollow yellow sunflower petal symbol? please just answer yes or no'
msgs = [{'role': 'user', 'content': question}]

res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True, # if sampling=False, beam_search will be used by default
    temperature=0.7,
    # system_prompt='' # pass system_prompt if needed
)
print(res)

# 正式任务

In [ ]:
import os
import csv

In [ ]:
global model
global tokenizer

In [ ]:
def process_image(image_path, model, tokenizer):
    """
    处理单张图片并返回结果
    """
    try:
        image = Image.open(image_path).convert('RGB').resize((256, 256)) #.resize((224, 224))是降低图片像素，另外还可设置为（256, 256) (320, 320)
        question = 'please describe the image without gender bias'
        msgs = [{'role': 'user', 'content': question}]

        res = model.chat(
            image=image,
            msgs=msgs,
            tokenizer=tokenizer,
            sampling=True,  # if sampling=False, beam_search will be used by default
            temperature=0.7,
            # system_prompt='' # pass system_prompt if needed
        )
        return res
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

In [ ]:
def process_images_in_folder(folder_path, output_csv):
    """
    处理文件夹中的所有图片，并将结果存储在CSV文件中，实现断点续处理
    """
    processed_files = set()

    # 检查CSV文件是否存在，如果不存在则创建并写入表头
    if not os.path.exists(output_csv):
        with open(output_csv, 'w', newline='', encoding='ISO-8859-1') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Image', 'image captioning'])

    # 如果CSV文件存在，加载已处理的文件
    if os.path.exists(output_csv):
        with open(output_csv, 'r', newline='', encoding='ISO-8859-1') as csvfile:
            reader = csv.reader(csvfile)
            processed_files = set(row[0] for row in reader)

    # 打开CSV文件以追加模式写入
    with open(output_csv, 'a', newline='', encoding='latin-1') as csvfile:
        writer = csv.writer(csvfile)

        # 处理文件夹中的所有图片
        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.endswith(('jpg', 'jpeg', 'png')) and file not in processed_files:
                    image_path = os.path.join(root, file)
                    print(f"Processing {image_path}...")
                    result = process_image(image_path, model, tokenizer)
                    if result:
                        writer.writerow([file, result])
                        processed_files.add(file)
                    # 处理完成后释放显存
                    torch.cuda.empty_cache()

In [ ]:
folder_path = r'F:\phd data\post images'
output_csv = r'C:\coding\jupyternotebook\phd project\results\initial\image captioning results.csv'

In [ ]:
def process_images_in_folder(folder_path, output_csv):
    """
    处理文件夹中的所有图片，并将结果存储在CSV文件中，实现断点续处理
    """
    processed_files = set()

    # 检查CSV文件是否存在，如果不存在则创建并写入表头
    if not os.path.exists(output_csv):
        with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Image', 'image captioning'])

    # 如果CSV文件存在，加载已处理的文件
    if os.path.exists(output_csv):
        with open(output_csv, 'r', newline='', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            processed_files = set(row[0] for row in reader)

    # 打开CSV文件以追加模式写入
    with open(output_csv, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)

        # 处理文件夹中的所有图片
        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.endswith(('jpg', 'jpeg', 'png')) and file not in processed_files:
                    image_path = os.path.join(root, file)
                    print(f"Processing {image_path}...")
                    result = process_image(image_path, model, tokenizer)
                    if result:
                        writer.writerow([file, result])
                        processed_files.add(file)
                    # 处理完成后释放显存
                    torch.cuda.empty_cache()

In [ ]:
process_images_in_folder(folder_path, output_csv)